In [1]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/models/tensorflow/bert/tensorflow2/en-uncased-l-12-h-768-a-12/3/saved_model.pb
/kaggle/input/models/tensorflow/bert/tensorflow2/en-uncased-l-12-h-768-a-12/3/assets/vocab.txt
/kaggle/input/models/tensorflow/bert/tensorflow2/en-uncased-l-12-h-768-a-12/3/variables/variables.index
/kaggle/input/models/tensorflow/bert/tensorflow2/en-uncased-l-12-h-768-a-12/3/variables/variables.data-00000-of-00001
/kaggle/input/models/tensorflow/bert/tensorflow2/en-uncased-preprocess/3/saved_model.pb
/kaggle/input/models/tensorflow/bert/tensorflow2/en-uncased-preprocess/3/keras_metadata.pb
/kaggle/input/models/tensorflow/bert/tensorflow2/en-uncased-preprocess/3/assets/vocab.txt
/kaggle/input/models/tensorflow/bert/tensorflow2/en-uncased-preprocess/3/variables/variables.index
/kaggle/input/models/tensorflow/bert/tensorflow2/en-uncased-preprocess/3/variables/variables.data-00000-of-00001
/kaggle/input/datasets/tmehul/spamcsv/spam.csv


In [3]:
import tensorflow as tf
import tensorflow_hub as hub
import tensorflow_text as text

2026-05-11 09:36:10.430113: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778492170.750438      16 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778492170.838843      16 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778492171.655818      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778492171.655861      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778492171.655863      16 computation_placer.cc:177] computation placer alr

In [4]:
import pandas as pd

df = pd.read_csv('/kaggle/input/datasets/tmehul/spamcsv/spam.csv',  encoding='latin-1')
df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [5]:
df1 = df.drop(['Unnamed: 2'	,'Unnamed: 3',	'Unnamed: 4'], axis='columns')
df1['Category'] = df['v1']
df1['Message'] = df['v2']

In [6]:
df1.drop(['v1','v2'], axis='columns',inplace= True)

In [7]:
df1.head()

,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [8]:
df1.groupby('Category').describe()

Message                                                            \
           count unique                                                top   
Category                                                                     
ham         4825   4516                             Sorry, I'll call later   
spam         747    653  Please call our customer service representativ...   

               
         freq  
Category       
ham        30  
spam        4

there is imbalance in the dataset

In [9]:
df1['spam'] = df1['Category'].apply(lambda c:1 if c=='spam' else 0)
df1

,Category,Message,spam
0,ham,"Go until jurong point, crazy.. Available only ...",0
1,ham,Ok lar... Joking wif u oni...,0
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,1
3,ham,U dun say so early hor... U c already then say...,0
4,ham,"Nah I don't think he goes to usf, he lives aro...",0
...,...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...,1
5568,ham,Will Ì_ b going to esplanade fr home?,0
5569,ham,"Pity, * was in mood for that. So...any other s...",0
5570,ham,The guy did some bitching but I acted like i'd...,0


In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df1['Message'], df1['spam'], test_size=0.2, stratify=df1['spam'])

In [11]:
y_train.value_counts()

spam
0    3859
1     598
Name: count, dtype: int64

In [12]:
y_test.value_counts()

spam
0    966
1    149
Name: count, dtype: int64

In [13]:
y_train.size

4457

In [14]:
encoder_url ="https://kaggle.com/models/tensorflow/bert/TensorFlow2/en-uncased-l-12-h-768-a-12/3"
preprocess_url = "https://kaggle.com/models/tensorflow/bert/TensorFlow2/en-uncased-preprocess/3"

In [15]:
bert_preprocess_model = hub.KerasLayer(preprocess_url)
bert_encode_layer = hub.KerasLayer(encoder_url)

2026-05-11 09:36:51.101895: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [16]:
def get_sentence_embedding(sentence):
    preprocessed_text = bert_preprocess_model(sentence)
    return bert_encode_layer(preprocessed_text)['pooled_output']

In [17]:
e =get_sentence_embedding(['Banana', 'grape', 'apple','jacobs', 'RATAN TATA'])

In [18]:
from sklearn.metrics.pairwise import cosine_similarity
cosine_similarity([e[0]], [e[1]])

array([[0.9718795]], dtype=float32)

In [19]:
cosine_similarity([e[1]], [e[-1]])

array([[0.9411289]], dtype=float32)

In [20]:
import tf_keras as keras
text_input = keras.layers.Input(shape=(), dtype=tf.string, name='text')
preprocessed_text = bert_preprocess_model(text_input)
outputs = bert_encode_layer(preprocessed_text)
l = tf.keras.layers.Dropout(0.1,name='dropout')(outputs['pooled_output'])
l = tf.keras.layers.Dense(1, activation='sigmoid',name='output')(l)

model = keras.Model(inputs=[text_input], outputs=[l])
model.summary()

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 text (InputLayer)           [(None,)]                    0         []                            
                                                                                                  
 keras_layer (KerasLayer)    {'input_type_ids': (None,    0         ['text[0][0]']                
                             128),                                                                
                              'input_mask': (None, 128)                                           
                             , 'input_word_ids': (None,                                           
                              128)}                                                               
                                                                                              

In [21]:
model.compile(
    optimizer = 'adam',
    loss = 'binary_crossentropy',
    metrics = 'accuracy'
)

In [22]:
model.evaluate(X_test, y_test)

35/35 [==============================] - 190s 5s/step - loss: 0.6441 - accuracy: 0.6377


[0.6440805792808533, 0.6376681327819824]

In [23]:
model.predict(['hey Amy,are you coming to the party letter'])

1/1 [==============================] - 1s 902ms/step


array([[0.41793042]], dtype=float32)